# Hybrid Search: BM25 + Semantic + RRF

**Companion notebook for *The Complementary Failure Modes* blog post**

---

This notebook implements every concept from the blog end-to-end on a small, inspectable corpus:

| Section | What you'll build |
|---|---|
| 1. The Failure Modes | BM25 vs semantic — side-by-side on the `ERR-4092` example |
| 2. Reciprocal Rank Fusion | RRF from scratch + blog's worked example + k-parameter visualisation |
| 3. Linear Score Combination | Min-max normalisation, weighted fusion, alpha sweep |
| 4. SPLADE | Conceptual demo using BERT masked-language modelling |
| 5. Vector DB Code | Weaviate / Qdrant / Pinecone patterns (ready to copy-paste) |
| 6. Full Comparison | nDCG@5 across all methods |

**Required packages:** `rank-bm25`, `sentence-transformers`, `numpy`, `matplotlib`, `scikit-learn`  
**Optional (SPLADE demo):** `transformers`, `torch`

In [ ]:
# Uncomment to install
# !pip install rank-bm25 sentence-transformers numpy matplotlib scikit-learn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('Setup complete.')

---
## 1. The Complementary Failure Modes

The blog's running example:

> **Query:** `ERR-4092 connection pool exhaustion workaround`

| Part of query | Best method | Why |
|---|---|---|
| `ERR-4092` (precise identifier) | BM25 | Rare token → high IDF weight, exact match |
| `connection pool exhaustion workaround` (concept) | Semantic | Meaning survives vocabulary mismatch |

We build a corpus designed to make both failure modes visible.

In [ ]:
# Corpus engineered to expose complementary failure modes
documents = [
    {'id': 'doc_0', 'title': 'ERR-4092: Connection Pool Exhaustion',
     'content': 'Error code ERR-4092 occurs when the connection pool is exhausted. '
                'To resolve ERR-4092, increase pool size or implement connection timeouts. '
                'This error is logged as POOL_EXHAUSTED in application logs.'},

    {'id': 'doc_1', 'title': 'Fixing Database Connection Limits',
     'content': 'When your application opens too many simultaneous connections, queries fail. '
                'Tune connection settings, implement connection reuse, and set timeouts. '
                'This guide covers database connection management best practices.'},

    {'id': 'doc_2', 'title': 'ERR-4091: Query Timeout',
     'content': 'Error code ERR-4091 indicates a query exceeded maximum execution time. '
                'Check query execution plans and add appropriate indexes.'},

    {'id': 'doc_3', 'title': 'Database Performance Tuning',
     'content': 'Optimize database performance by tuning memory settings, analyzing query plans, '
                'and managing index fragmentation. Regular maintenance prevents degradation.'},

    {'id': 'doc_4', 'title': 'Too Many Open Database Connections',
     'content': 'Applications struggle with managing database connections at scale. '
                'When connections exceed server limits, new requests are rejected. '
                'Implement bounded connection pooling to prevent resource exhaustion.'},

    {'id': 'doc_5', 'title': 'ERR-4092 Root Cause Analysis',
     'content': 'A deep dive into ERR-4092: manifests under high concurrent load. '
                'Root cause is unbounded connection creation without a pool governor. '
                'Workaround: set max_pool_size and implement retry with exponential backoff.'},

    {'id': 'doc_6', 'title': 'Network Configuration Guide',
     'content': 'Configure network settings for optimal throughput. '
                'Covers TCP keepalive, socket buffer sizes, and interface configuration.'},

    {'id': 'doc_7', 'title': 'Workaround for Database Connection Saturation',
     'content': 'For connection saturation, implement exponential backoff on retry, '
                'use read replicas to distribute load, and enable connection multiplexing. '
                'These techniques reduce peak connection demand and prevent exhaustion.'},
]

# Ground-truth relevance grades (2=highly relevant, 1=relevant, 0=not relevant)
# For query: 'ERR-4092 connection pool exhaustion workaround'
relevance = {
    'doc_0': 2,  # ERR-4092 + connection pool — perfect match
    'doc_5': 2,  # ERR-4092 + workaround — perfect match
    'doc_7': 1,  # workaround for connection saturation — relevant
    'doc_1': 1,  # connection limit fixing — relevant
    'doc_4': 1,  # connection pooling — relevant
    'doc_2': 0,  # different error code — not relevant
    'doc_3': 0,  # general performance — not relevant
    'doc_6': 0,  # networking — not relevant
}

query = 'ERR-4092 connection pool exhaustion workaround'
doc_id_to_idx = {doc['id']: i for i, doc in enumerate(documents)}

print(f'Query: {query!r}\n')
labels = {2: 'HIGHLY RELEVANT', 1: 'relevant      ', 0: 'not relevant  '}
for doc in documents:
    doc_id = doc['id']
    title = doc['title']
    label = labels[relevance[doc_id]]
    print(f'  [{label}] [{doc_id}] {title}')

### 1a. BM25 — The Exact-Match Expert

In [ ]:
from rank_bm25 import BM25Okapi

def tokenize(text):
    return text.lower().split()

tokenized_corpus = [tokenize(doc['title'] + ' ' + doc['content']) for doc in documents]
bm25 = BM25Okapi(tokenized_corpus)

bm25_raw = bm25.get_scores(tokenize(query))
bm25_ranked = sorted(enumerate(bm25_raw), key=lambda x: x[1], reverse=True)

print('BM25 results for:', repr(query))
print(f'{"Rank":<5} {"Score":<8} {"Relevance":<12} Title')
print('-' * 65)
for rank, (idx, score) in enumerate(bm25_ranked, 1):
    doc_id = documents[idx]['id']
    title = documents[idx]['title']
    rel = relevance[doc_id]
    marker = '✓✓' if rel == 2 else ('✓' if rel == 1 else '✗')
    print(f'#{rank:<4} {score:<8.3f} {marker:<12} {title}')

### 1b. Semantic Search — The Conceptual Reasoner

In [ ]:
from sentence_transformers import SentenceTransformer, util

print('Loading all-MiniLM-L6-v2 (downloads ~90 MB on first run)...')
model = SentenceTransformer('all-MiniLM-L6-v2')

doc_texts = [doc['title'] + '. ' + doc['content'] for doc in documents]
doc_embeddings = model.encode(doc_texts, convert_to_tensor=True)
query_embedding = model.encode(query, convert_to_tensor=True)

semantic_raw = util.cos_sim(query_embedding, doc_embeddings)[0].numpy()
semantic_ranked = sorted(enumerate(semantic_raw), key=lambda x: x[1], reverse=True)

print('\nSemantic results for:', repr(query))
print(f'{"Rank":<5} {"Score":<8} {"Relevance":<12} Title')
print('-' * 65)
for rank, (idx, score) in enumerate(semantic_ranked, 1):
    doc_id = documents[idx]['id']
    title = documents[idx]['title']
    rel = relevance[doc_id]
    marker = '✓✓' if rel == 2 else ('✓' if rel == 1 else '✗')
    print(f'#{rank:<4} {score:<8.3f} {marker:<12} {title}')

In [ ]:
# Side-by-side visualisation of the failure modes
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

def rel_color(doc_id):
    r = relevance[doc_id]
    return '#2E7D32' if r == 2 else ('#66BB6A' if r == 1 else '#EF9A9A')

top_n = 6
y_pos = list(range(top_n - 1, -1, -1))

for ax, ranked, scores_arr, xlabel, title_str in [
    (axes[0], bm25_ranked, bm25_raw,     'BM25 Score',       'BM25 Results'),
    (axes[1], semantic_ranked, semantic_raw, 'Cosine Similarity', 'Semantic Search Results'),
]:
    top = ranked[:top_n]
    vals   = [s for _, s in top]
    colors = [rel_color(documents[i]['id']) for i, _ in top]
    labels = [f'[{documents[i]["id"]}] {documents[i]["title"][:32]}' for i, _ in top]

    ax.barh(y_pos, vals, color=colors)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel(xlabel)
    ax.set_title(title_str, fontweight='bold', fontsize=13)

legend_handles = [
    mpatches.Patch(color='#2E7D32', label='Highly Relevant (grade 2)'),
    mpatches.Patch(color='#66BB6A', label='Relevant (grade 1)'),
    mpatches.Patch(color='#EF9A9A', label='Not Relevant (grade 0)'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=3,
           fontsize=10, bbox_to_anchor=(0.5, -0.06))
fig.suptitle(f'Query: {query!r}', fontsize=11, style='italic', y=1.02)
plt.tight_layout()
plt.show()

print('Observation: BM25 correctly surfaces ERR-4092 docs (exact token match),')
print('but may bury purely conceptual matches like doc_7 (no literal "ERR-4092").')
print('Semantic search surfaces conceptual matches, but error codes are arbitrary')
print('strings with no embedding-space meaning.')

---
## 2. Reciprocal Rank Fusion (RRF)

**Core insight:** Don't compare scores (they're on incompatible scales). Compare *ranks*.

$$\text{RRF}(d) = \sum_{i} \frac{1}{k + \text{rank}_i(d)}$$

- $\text{rank}_i(d)$ = position of document $d$ in result list $i$ (1-indexed)
- $k$ = smoothing constant (default **60**, from Cormack et al. 2009)
- Documents absent from a list contribute **zero** (not penalised with a large rank)

Documents that appear in **both** lists get boosted — agreement between retrievers is a strong relevance signal.

In [ ]:
def reciprocal_rank_fusion(ranked_lists, k=60):
    """
    Fuse multiple ranked result lists using Reciprocal Rank Fusion.

    Args:
        ranked_lists: list of lists, each [(doc_id, score), ...] in rank order
        k:            RRF constant (default 60)

    Returns:
        [(doc_id, rrf_score), ...] sorted descending by RRF score
    """
    scores = defaultdict(float)
    for ranked_list in ranked_lists:
        for rank, (doc_id, _) in enumerate(ranked_list, start=1):
            scores[doc_id] += 1.0 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

print('reciprocal_rank_fusion() defined.')
print()
print('Formula: RRF(doc) = sum over lists of  1 / (k + rank_in_list)')
print(f'With k=60: rank #1 → {1/61:.5f}  |  rank #5 → {1/65:.5f}  |  rank #20 → {1/80:.5f}')

### 2a. Blog's Worked Example (step-by-step)

Reproducing the exact table from the blog:

In [ ]:
# Worked example from the blog
# BM25 top 5:     A(1) B(2) C(3) E(4) F(5)
# Semantic top 5: D(1) B(2) G(3) A(4) C(5)

bm25_example     = [('Doc A', 14.7), ('Doc B', 11.2), ('Doc C', 8.9), ('Doc E', 6.3), ('Doc F', 4.1)]
semantic_example = [('Doc D', 0.91), ('Doc B', 0.87), ('Doc G', 0.82), ('Doc A', 0.79), ('Doc C', 0.71)]

bm25_ranks = {d: r + 1 for r, (d, _) in enumerate(bm25_example)}
sem_ranks  = {d: r + 1 for r, (d, _) in enumerate(semantic_example)}
all_docs   = sorted(set(bm25_ranks) | set(sem_ranks))

k = 60
rows = []
for doc in all_docs:
    br = bm25_ranks.get(doc)
    sr = sem_ranks.get(doc)
    bc = 1 / (k + br) if br else 0.0
    sc = 1 / (k + sr) if sr else 0.0
    rows.append((doc, br, sr, bc, sc, bc + sc))

rows.sort(key=lambda x: x[5], reverse=True)

print('=' * 80)
print('RRF WORKED EXAMPLE  (k = 60)')
print('=' * 80)
print(f'{"Doc":<8} {"BM25 rank":<11} {"Sem rank":<11} {"BM25 contrib":<16} {"Sem contrib":<16} {"RRF Score"}')
print('-' * 80)
for doc, br, sr, bc, sc, total in rows:
    b_str = f'#{br}  1/(60+{br})={bc:.5f}' if br else 'absent → 0.00000'
    s_str = f'#{sr}  {sc:.5f}'              if sr else 'absent → 0.00000'
    in_both = ' ← both lists' if br and sr else ''
    print(f'{doc:<8} {str(br) if br else "—":<11} {str(sr) if sr else "—":<11} {bc:.5f}          {sc:.5f}          {total:.5f}{in_both}')

print()
print('Final RRF ranking:')
rrf_example = reciprocal_rank_fusion([bm25_example, semantic_example], k=60)
for rank, (doc, score) in enumerate(rrf_example, 1):
    in_both = ' ← appears in BOTH lists' if doc in bm25_ranks and doc in sem_ranks else ''
    print(f'  #{rank}: {doc}  (RRF score = {score:.5f}){in_both}')

### 2b. Understanding the `k` Parameter

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: score contribution 1/(k+rank) for different k values
ranks = np.arange(1, 21)
k_values = [1, 10, 30, 60, 120]
palette  = ['#E53935', '#FB8C00', '#43A047', '#1E88E5', '#8E24AA']

for k_val, color in zip(k_values, palette):
    axes[0].plot(ranks, 1 / (k_val + ranks), marker='o', markersize=4,
                 label=f'k={k_val}', color=color)

axes[0].set_xlabel('Rank Position')
axes[0].set_ylabel('Score contribution  1/(k + rank)')
axes[0].set_title('Effect of k on Rank Weighting', fontweight='bold')
axes[0].legend()
axes[0].set_xticks(ranks[::2])

# Right: how much does rank #1 beat rank #5?
k_range = np.arange(1, 201)
ratio   = (1 / (k_range + 1)) / (1 / (k_range + 5))

axes[1].plot(k_range, ratio, color='#1E88E5', linewidth=2)
axes[1].axhline(y=1.0,  color='grey',    linestyle='--', alpha=0.6, label='No preference (ratio=1)')
axes[1].axvline(x=60,   color='#E53935', linestyle='--', alpha=0.8, label='k=60 (default)')
axes[1].fill_between(k_range, 1, ratio, alpha=0.12, color='#1E88E5')
axes[1].set_xlabel('k value')
axes[1].set_ylabel('Score(rank 1) / Score(rank 5)')
axes[1].set_title('How much does being #1 matter vs #5?', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

print('Small k  → being #1 matters enormously (winner-takes-all behaviour)')
print('Large k  → being in the list at all is what matters (rank position smoothed out)')
print()
print(f'At k=60 : rank #1 scores {(1/61)/(1/65):.3f}x more than rank #5   — top ranks still matter but gently')
print(f'At k=1  : rank #1 scores {(1/2) /(1/6) :.1f}x  more than rank #5   — being #1 dominates')

### 2c. Apply RRF to Our Corpus

In [ ]:
# Convert index-based ranked lists to (doc_id, score) lists for RRF
bm25_id_ranked     = [(documents[i]['id'], s) for i, s in bm25_ranked]
semantic_id_ranked = [(documents[i]['id'], s) for i, s in semantic_ranked]

hybrid_results = reciprocal_rank_fusion([bm25_id_ranked, semantic_id_ranked], k=60)

print(f'Query: {query!r}\n')
print(f'{"Rank":<5} {"RRF Score":<12} {"Rel":<6} {"BM25 rank":<11} {"Sem rank":<11} Title')
print('-' * 80)

bm25_rank_map = {documents[i]['id']: r + 1 for r, (i, _) in enumerate(bm25_ranked)}
sem_rank_map  = {documents[i]['id']: r + 1 for r, (i, _) in enumerate(semantic_ranked)}

for rank, (doc_id, rrf_score) in enumerate(hybrid_results, 1):
    idx   = doc_id_to_idx[doc_id]
    title = documents[idx]['title']
    rel   = relevance[doc_id]
    br    = bm25_rank_map[doc_id]
    sr    = sem_rank_map[doc_id]
    marker = '✓✓' if rel == 2 else ('✓' if rel == 1 else '✗')
    print(f'#{rank:<4} {rrf_score:<12.5f} {marker:<6} #{br:<10} #{sr:<10} {title}')

---
## 3. Linear Score Combination (Weighted Fusion)

$$\text{hybrid}(d) = \alpha \cdot \text{norm}(\text{BM25}(d)) + (1-\alpha) \cdot \text{norm}(\text{semantic}(d))$$

**Problem:** BM25 scores and cosine similarities are on completely different scales.

```
BM25 scores:         [14.7, 11.2, 8.9, 6.3, 4.1]   ← range: ~0–30
Cosine similarities: [0.87, 0.82, 0.79, 0.71, 0.65] ← range: -1 to 1
```

Adding raw scores means BM25 dominates every time. **Normalise first.**

In [ ]:
def minmax_normalize(scores):
    """Per-query min-max normalisation to [0, 1]."""
    mn, mx = scores.min(), scores.max()
    if mx == mn:
        return np.zeros_like(scores)
    return (scores - mn) / (mx - mn)

def zscore_normalize(scores):
    """Z-score normalisation: (x - mean) / std."""
    mu, sigma = scores.mean(), scores.std()
    if sigma == 0:
        return np.zeros_like(scores)
    return (scores - mu) / sigma

# Why normalisation matters — raw vs normalised
bm25_scores_np   = np.array(bm25_raw)
semantic_scores_np = np.array(semantic_raw)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

x = np.arange(len(documents))
doc_labels = [doc['id'] for doc in documents]

# Raw scores — BM25 dwarfs cosine
axes[0].bar(x - 0.2, bm25_scores_np,    width=0.4, label='BM25 raw',    color='#1E88E5', alpha=0.8)
axes[0].bar(x + 0.2, semantic_scores_np, width=0.4, label='Cosine raw',  color='#43A047', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(doc_labels, rotation=45)
axes[0].set_title('Raw Scores — BM25 dominates the scale', fontweight='bold')
axes[0].legend()

# Normalised — now comparable
axes[1].bar(x - 0.2, minmax_normalize(bm25_scores_np),    width=0.4, label='BM25 normalised',   color='#1E88E5', alpha=0.8)
axes[1].bar(x + 0.2, minmax_normalize(semantic_scores_np), width=0.4, label='Cosine normalised', color='#43A047', alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(doc_labels, rotation=45)
axes[1].set_title('Min-Max Normalised — now on the same [0,1] scale', fontweight='bold')
axes[1].legend()

plt.suptitle('Why normalisation is required for linear combination', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
def linear_hybrid(bm25_scores, semantic_scores, alpha, normalize_fn=minmax_normalize):
    """
    Linear score combination.
    alpha=1 → pure BM25 ; alpha=0 → pure semantic
    """
    norm_bm25 = normalize_fn(np.array(bm25_scores))
    norm_sem  = normalize_fn(np.array(semantic_scores))
    return alpha * norm_bm25 + (1 - alpha) * norm_sem


def ndcg_at_k(ranked_doc_ids, relevance_dict, k=5):
    """Compute nDCG@k given a ranked list and relevance grades."""
    def dcg(rels):
        return sum(r / np.log2(i + 2) for i, r in enumerate(rels))

    rels = [relevance_dict.get(doc_id, 0) for doc_id in ranked_doc_ids[:k]]
    ideal_rels = sorted(relevance_dict.values(), reverse=True)[:k]
    idcg = dcg(ideal_rels)
    return dcg(rels) / idcg if idcg > 0 else 0.0


# Sweep alpha from 0 (pure semantic) to 1 (pure BM25)
alphas = np.arange(0.0, 1.05, 0.05)
ndcg_scores = []

for alpha in alphas:
    hybrid_scores = linear_hybrid(bm25_raw, semantic_raw, alpha)
    ranked_ids = [documents[i]['id'] for i in np.argsort(hybrid_scores)[::-1]]
    ndcg_scores.append(ndcg_at_k(ranked_ids, relevance, k=5))

best_alpha = alphas[np.argmax(ndcg_scores)]
best_ndcg  = max(ndcg_scores)

# Also compute baselines
bm25_only_ids = [documents[i]['id'] for i, _ in bm25_ranked]
sem_only_ids  = [documents[i]['id'] for i, _ in semantic_ranked]
rrf_ids       = [doc_id for doc_id, _ in hybrid_results]

ndcg_bm25 = ndcg_at_k(bm25_only_ids,  relevance, k=5)
ndcg_sem  = ndcg_at_k(sem_only_ids,   relevance, k=5)
ndcg_rrf  = ndcg_at_k(rrf_ids,        relevance, k=5)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(alphas, ndcg_scores, color='#1E88E5', linewidth=2.5, label='Linear combination (min-max normalised)')
ax.axvline(x=best_alpha, color='#1E88E5', linestyle='--', alpha=0.7, label=f'Best alpha = {best_alpha:.2f}')
ax.axhline(y=ndcg_bm25,  color='#FB8C00', linestyle=':',  linewidth=2, label=f'BM25 only (nDCG@5={ndcg_bm25:.3f})')
ax.axhline(y=ndcg_sem,   color='#43A047', linestyle=':',  linewidth=2, label=f'Semantic only (nDCG@5={ndcg_sem:.3f})')
ax.axhline(y=ndcg_rrf,   color='#E53935', linestyle='-',  linewidth=2, label=f'RRF hybrid (nDCG@5={ndcg_rrf:.3f})')
ax.set_xlabel('Alpha  (0 = pure semantic,  1 = pure BM25)')
ax.set_ylabel('nDCG@5')
ax.set_title('Linear Combination: Alpha Sweep vs Baselines', fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

print(f'Best linear combination: alpha={best_alpha:.2f}, nDCG@5={best_ndcg:.3f}')
print(f'RRF hybrid (no tuning):            nDCG@5={ndcg_rrf:.3f}')
print()
print('Rule of thumb: if you lack labelled data, use RRF.')
print('Linear combination can edge ahead when alpha is tuned on your domain.')

---
## 4. SPLADE — Learned Sparse Retrieval

**Key idea:** Instead of two separate systems + fusion, train a single model that produces
**sparse** vectors (like BM25) but with **learned weights** that capture semantic meaning.

| Property | BM25 | Dense Embeddings | SPLADE |
|---|---|---|---|
| Vector format | Sparse (term counts) | Dense (e.g. 384-d) | Sparse (vocab-size) |
| Semantic understanding | ✗ | ✓ | ✓ |
| Uses inverted index | ✓ | ✗ | ✓ |
| Requires training | ✗ | ✓ (fine-tune) | ✓ (MS MARCO etc.) |
| Inference at index time | ✗ | ✓ | ✓ |

SPLADE performs **neural document expansion**: terms absent from the document can receive
non-zero weights if the model judges them semantically relevant.

The cell below uses BERT's masked-language-model head to *demonstrate this expansion intuition*.

In [ ]:
# SPLADE conceptual demo — BERT masked LM as a proxy for term expansion
# Requires: pip install transformers torch

try:
    from transformers import pipeline

    fill_mask = pipeline('fill-mask', model='bert-base-uncased')

    # Simulate what SPLADE does: for a document about connection pools,
    # what *other* terms does the model associate with this context?
    test_sentences = [
        'The application experienced [MASK] due to too many open database connections.',
        'To fix connection pool [MASK], increase the maximum pool size.',
        'ERR-4092 is a [MASK] error that occurs under heavy concurrent load.',
    ]

    for sentence in test_sentences:
        results = fill_mask(sentence, top_k=6)
        predicted = [r['token_str'] for r in results]
        print(f'Context: {sentence}')
        print(f'Predicted terms: {predicted}')
        print()

    print('This is the SPLADE intuition: the model expands documents with semantically')
    print('related terms even when those terms do not literally appear in the text.')
    print('SPLADE does this for EVERY position in the vocabulary simultaneously,')
    print('producing a sparse weight vector over the full vocab (~30,000 tokens).')

except ImportError:
    print('transformers not installed. Install with: pip install transformers torch')
    print()
    print('Conceptual SPLADE output for a document about connection pool exhaustion:')
    print()
    print('  Document text:  "ERR-4092 occurs when the connection pool is exhausted"')
    print()
    print('  SPLADE sparse vector (non-zero entries only):')
    splade_mock = [
        ('connection',   0.94),
        ('pool',         0.89),
        ('exhausted',    0.87),
        ('ERR',          0.85),
        ('4092',         0.84),
        # --- terms NOT in document, but expanded ---
        ('database',     0.71),
        ('timeout',      0.68),
        ('saturation',   0.62),
        ('workaround',   0.58),
        ('overflow',     0.52),
        ('limit',        0.49),
        ('capacity',     0.41),
    ]
    for term, weight in splade_mock:
        in_doc = term.lower() in 'err-4092 occurs when the connection pool is exhausted'
        marker = '  (in document)' if in_doc else '  ← EXPANDED by model'
        print(f'  {term:<16} weight={weight:.2f}{marker}')

---
## 5. Vector Database Implementations

All major vector databases support hybrid search natively. The pattern is always:

1. **Index time:** store dense vector (embedding) + sparse representation (BM25 terms or SPLADE) per document
2. **Query time:** generate both representations for the query
3. **Retrieve:** run both searches, fuse (usually RRF), return merged list

The cells below show the API patterns — swap in your connection details to use them.

In [ ]:
# ── WEAVIATE ──────────────────────────────────────────────────────────────────
# Uses linear combination (alpha parameter).
# alpha=0 → pure BM25,  alpha=1 → pure vector  (note: inverse of our convention)
#
# pip install weaviate-client

weaviate_example = '''
import weaviate

client = weaviate.Client("http://localhost:8080")

result = (
    client.query
    .get("Document", ["title", "content"])
    .with_hybrid(
        query="ERR-4092 connection pool exhaustion",
        alpha=0.5,   # 0 = pure BM25, 1 = pure vector, 0.5 = equal blend
    )
    .with_limit(10)
    .do()
)
'''
print('Weaviate hybrid search:')
print(weaviate_example)

In [ ]:
# ── QDRANT ────────────────────────────────────────────────────────────────────
# Explicit prefetch + fusion step. Supports RRF natively.
#
# pip install qdrant-client

qdrant_example = '''
from qdrant_client import QdrantClient, models

client = QdrantClient("localhost", port=6333)

results = client.query_points(
    collection_name="documents",
    prefetch=[
        models.Prefetch(
            query=models.SparseVector(indices=[...], values=[...]),  # BM25 / SPLADE
            using="sparse",
            limit=20,   # retrieve more candidates than final top_k
        ),
        models.Prefetch(
            query=[0.01, 0.45, ...],  # dense embedding
            using="dense",
            limit=20,
        ),
    ],
    query=models.FusionQuery(fusion=models.Fusion.RRF),  # RRF fusion
    limit=10,
)
# prefetch limit >> final limit to give RRF enough candidates to work with
'''
print('Qdrant hybrid search with RRF:')
print(qdrant_example)

In [ ]:
# ── PINECONE ──────────────────────────────────────────────────────────────────
# Dense + sparse in a single index. Fusion handled internally.
# sparse_vector indices = token IDs, values = BM25/SPLADE weights
#
# pip install pinecone-client

pinecone_example = '''
from pinecone import Pinecone

pc    = Pinecone(api_key="YOUR_API_KEY")
index = pc.Index("my-index")

results = index.query(
    vector=[0.01, 0.45, ...],          # dense embedding
    sparse_vector={                     # sparse (keyword) signal
        "indices": [102, 5743, 8821],   # token IDs
        "values":  [0.5,  0.8,  0.3],   # BM25 or SPLADE weights
    },
    top_k=10,
    alpha=0.5,  # 0 = pure sparse, 1 = pure dense
)
'''
print('Pinecone hybrid search:')
print(pinecone_example)

---
## 6. Full Method Comparison

Evaluating all methods on our corpus using **nDCG@5** (Normalised Discounted Cumulative Gain).

nDCG@5 rewards:
- Highly relevant documents ranked first
- Penalties for irrelevant documents surfacing at the top
- Range: 0 (worst) → 1.0 (perfect)

In [ ]:
# ─── Evaluate all methods ──────────────────────────────────────────────────────

# Best linear combination (alpha from the sweep above)
best_hybrid_scores = linear_hybrid(bm25_raw, semantic_raw, best_alpha)
best_linear_ids    = [documents[i]['id'] for i in np.argsort(best_hybrid_scores)[::-1]]
ndcg_linear        = ndcg_at_k(best_linear_ids, relevance, k=5)

# RRF with k=1 and k=120 to show sensitivity
rrf_k1   = reciprocal_rank_fusion([bm25_id_ranked, semantic_id_ranked], k=1)
rrf_k120 = reciprocal_rank_fusion([bm25_id_ranked, semantic_id_ranked], k=120)
ndcg_rrf_k1   = ndcg_at_k([d for d, _ in rrf_k1],   relevance, k=5)
ndcg_rrf_k120 = ndcg_at_k([d for d, _ in rrf_k120], relevance, k=5)

methods = [
    ('BM25 only',                  ndcg_bm25,      '#FB8C00'),
    ('Semantic only',              ndcg_sem,        '#43A047'),
    ('Linear combo (best alpha)',  ndcg_linear,     '#AB47BC'),
    ('RRF  k=1',                   ndcg_rrf_k1,     '#90CAF9'),
    ('RRF  k=60  (default)',       ndcg_rrf,        '#E53935'),
    ('RRF  k=120',                 ndcg_rrf_k120,   '#FFCDD2'),
]

fig, ax = plt.subplots(figsize=(11, 5))

names  = [m[0] for m in methods]
scores = [m[1] for m in methods]
colors = [m[2] for m in methods]

bars = ax.barh(range(len(methods)), scores, color=colors, edgecolor='white', linewidth=0.5)
ax.set_yticks(range(len(methods)))
ax.set_yticklabels(names, fontsize=11)
ax.set_xlabel('nDCG@5  (higher is better)', fontsize=12)
ax.set_title('Retrieval Quality Comparison', fontweight='bold', fontsize=13)
ax.set_xlim(0, 1.1)
ax.axvline(x=1.0, color='grey', linestyle='--', alpha=0.4, label='Perfect score')

for bar, score in zip(bars, scores):
    ax.text(score + 0.01, bar.get_y() + bar.get_height() / 2,
            f'{score:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

print('Results summary:')
for name, score, _ in sorted(methods, key=lambda x: x[1], reverse=True):
    print(f'  {name:<35} nDCG@5 = {score:.3f}')

In [ ]:
# ─── Show the top-5 ranked results for each method ─────────────────────────────

print(f'Top-5 results for each method  |  Query: {query!r}\n')
print(f'{"Rank":<6} {"BM25 only":<32} {"Semantic only":<32} {"RRF (k=60)":<32} {"Linear best"}')
print('-' * 120)

def short(doc_id):
    title = documents[doc_id_to_idx[doc_id]]['title']
    return f'[{doc_id}] {title}'[:30]

for rank in range(5):
    bm25_id   = documents[bm25_ranked[rank][0]]['id']
    sem_id    = documents[semantic_ranked[rank][0]]['id']
    rrf_id    = hybrid_results[rank][0]
    lin_id    = best_linear_ids[rank]

    def fmt(doc_id):
        rel = relevance[doc_id]
        marker = '✓✓' if rel == 2 else ('✓' if rel == 1 else '✗')
        title = documents[doc_id_to_idx[doc_id]]['title'][:26]
        return f'{marker} [{doc_id}] {title}'

    print(f'#{rank+1:<5} {fmt(bm25_id):<32} {fmt(sem_id):<32} {fmt(rrf_id):<32} {fmt(lin_id)}')

---
## Key Takeaways

### The Decision Framework

| Situation | Recommendation |
|---|---|
| No labelled data, shipping today | **RRF with k=60** |
| Have labelled data, want to squeeze more | **Linear combo** — tune alpha on your domain |
| Already running neural models, want single-index elegance | **SPLADE** |
| Query-heavy with very precise identifiers (codes, SKUs, IDs) | Ensure BM25 weight is high |
| Mostly natural-language queries, rich semantic variation | Ensure semantic weight is high |

### Why RRF Is the Right Default

1. **Scale-invariant** — doesn't matter that BM25 scores range 0–30 and cosine 0–1
2. **No normalisation** — rank-based fusion just works
3. **Extensible** — add a third retriever by adding a term to the sum
4. **No training** — deploy today without any labelled data
5. **Debuggable** — inspect each retriever's ranked list independently

### The One-Liner to Remember

> **BM25 + semantic + RRF is almost always better than either method alone. Start there.**

---

*Up next: Neural Rerankers — using a cross-encoder to deeply analyse query↔document relevance on the shortlist RRF gives you.*